# 05 — o200k Tokenizer Measurement (D-04, canonical)

> **Phase 4 · SSOT §12.4 D-04 Token Measurement**
> **Canonical artifact**: `data/registry/TOKEN_O200K_BASE_v001.parquet` (SSOT §38 naming)

> **Lineage authority**: Claude-B independent forensic adjudication
> `441d5802bfebe178fd220d08b653c60dfad17faf`
> `ssot/2026-08-17_1730_KOEN_TP_G2_G3_G4_FINAL_ADJUDICATION.md`
> `G2_REPRESENTATION_INTEGRITY_PASS` · `G3_TOKENIZER_INTEGRITY_PASS` · `G4_MORPHOLOGY_INTEGRITY_PASS`
> `MEASUREMENT_FOUNDATION_CLOSED_THROUGH_G4`
>
> Every artifact identity below is pinned to values Claude-B recomputed from the physical Parquet,
> not to values this pipeline reported about itself.


## Scope

Pair-level token counts under the frozen `o200k_base` encoding, and the exact multiplicative
decomposition of the Tokenization Premium defined in SSOT §8:

```
TP    = CodePointRatio · ByteDensityRatio · CompressionPenalty
logTP = logCodePointRatio + logByteDensityRatio + logCompressionPenalty
```

Track A rules from SSOT §14.2 apply: no chat template, no special tokens, only `*_text_analysis` is
encoded, and every pair is round-tripped.

This notebook measures. It draws **no** inference: RQ1's signed-rank test on `median(logTP)` is
NB08's work, and the regex chunk mechanism is NB06's.

## Status of the earlier scaffold

The previous NB05 was a synthetic-only, four-row scaffold with the full population deferred. That
state is superseded: the full cohort has been measured and this notebook now validates the resulting
artifact.

## Fail-closed lineage

This notebook **validates and reuses**; it does not rebuild. Canonical inputs are asserted through
`tokenization_premium.lineage.assert_canonical_artifact`, which checks path, SHA-256, column count,
row count and identifier uniqueness, and raises `CanonicalArtifactIdentityMismatch` on any
difference. There is no `if exists()` guard and no descent to a pilot, synthetic, or earlier
version: a missing or altered artifact stops the notebook rather than silently changing what is
being reported.

## 0 — Environment and rebuild gate

In [1]:
from __future__ import annotations

import json

import duckdb
import pandas as pd

from tokenization_premium.lineage import (
    ADJUDICATION_COMMIT,
    ADJUDICATION_DOC,
    CANONICAL_ARTIFACTS,
    CANONICAL_PAIR_SET_MD5,
    assert_canonical_artifact,
    describe_historical,
)
from tokenization_premium.paths import PROJECT_ROOT

pd.set_option("display.width", 200, "display.max_columns", 50)


# 모든 전집단 질의는 DuckDB pushdown으로 처리해 메모리를 bounded 상태로 유지한다.
def connect() -> duckdb.DuckDBPyConnection:
    con = duckdb.connect()
    con.execute("SET memory_limit='5GB'")
    con.execute("SET threads=8")
    con.execute("SET preserve_insertion_order=false")
    spill = PROJECT_ROOT / ".runtime" / "canonical-nb" / "duckdb-spill"
    spill.mkdir(parents=True, exist_ok=True)
    con.execute(f"SET temp_directory='{spill.as_posix()}'")
    return con


CON = connect()
print(f"lineage authority : {ADJUDICATION_COMMIT[:12]}  {ADJUDICATION_DOC}")

lineage authority : 441d5802bfeb  ssot/2026-08-17_1730_KOEN_TP_G2_G3_G4_FINAL_ADJUDICATION.md


In [2]:
REBUILD_CANONICAL_ARTIFACT = False   # Director-authorized rebuild only; Run All must never regenerate

if REBUILD_CANONICAL_ARTIFACT:
    raise RuntimeError(
        "REBUILD_CANONICAL_ARTIFACT=True는 Director 승인 실행에서만 사용한다. "
        "기본 Run All은 canonical artifact를 재생성하지 않고 검증·재사용만 한다."
    )
print("rebuild gate: DISABLED (validate/reuse only)")

rebuild gate: DISABLED (validate/reuse only)


## 1 — Canonical artifact identity (fail-closed)

The D-02 input and the D-04 output are both asserted, including the sorted pair-set hash Claude-B
recomputed independently.

Note on the column count: the artifact, its manifest and the schema builder all carry **28** columns.
A `29` appears in the body of commit `92dc07a`; Claude-B adjudicated that as `REPORT_TYPO_ONLY` and
it must not be propagated.

In [3]:
REP = assert_canonical_artifact("REP_FEATURES_v002", verify_pair_set=True, con=CON)
TOK = assert_canonical_artifact("TOKEN_O200K_BASE_v001", verify_pair_set=True, con=CON)

for confirmed in (REP, TOK):
    print(f"{confirmed['name']:26s} {confirmed['identity']}")
    print(f"   sha256 {confirmed['sha256']}")
    print(f"   rows   {confirmed['row_count']:>9,}   columns {confirmed['column_count']:>3}"
          f"   distinct {confirmed['id_column']} {confirmed['distinct_id']:,}")
    print(f"   pair-set md5 {confirmed['pair_set_md5']}")
assert REP["pair_set_md5"] == TOK["pair_set_md5"] == CANONICAL_PAIR_SET_MD5

T = f"read_parquet('{CANONICAL_ARTIFACTS['TOKEN_O200K_BASE_v001'].path.as_posix()}')"
N = TOK["row_count"]
print(f"\ncohort N = {N:,} (derived from the artifact, not hard-coded)")
print(f"column count = {TOK['column_count']} (28 is authoritative; the 29 in commit 92dc07a "
      "is REPORT_TYPO_ONLY)")

REP_FEATURES_v002          CANONICAL_ARTIFACT_IDENTITY_VERIFIED
   sha256 dfae8e01cd3fe2ca949d8754678e508203ad1a7aa6abea418008a33ac650d309
   rows   3,835,988   columns  49   distinct pair_id 3,835,988
   pair-set md5 d9660d654ee449e4d0c23a0070225274
TOKEN_O200K_BASE_v001      CANONICAL_ARTIFACT_IDENTITY_VERIFIED
   sha256 1c30e3276222dd94885ae4f79fc6ab5c45e4e26226de0afd91fc6b1f7d2c16e7
   rows   3,835,988   columns  28   distinct measurement_id 3,835,988
   pair-set md5 d9660d654ee449e4d0c23a0070225274

cohort N = 3,835,988 (derived from the artifact, not hard-coded)
column count = 28 (28 is authoritative; the 29 in commit 92dc07a is REPORT_TYPO_ONLY)


## 2 — Tokenizer provenance freeze (SSOT §31 G3)

G3 requires the tiktoken version and the `pat_str` hash to be recorded. The encoding is loaded from
the offline cache only — no network fallback — and the five frozen hashes are compared three ways:
the Phase-0 freeze, the live module constants, and the values embedded in every artifact row.

In [4]:
from tokenization_premium.tokenizer_measurement import (
    ENCODING_FILE_SHA256,
    MERGEABLE_RANKS_HASH,
    PAT_STR_SHA256,
    SPECIAL_TOKENS_HASH,
    TIKTOKEN_VERSION,
    TOKENIZER_CONFIG,
    TOKENIZER_CONFIG_SHA256,
    TOKENIZER_ID,
    load_o200k_base_offline,
)

CACHE = PROJECT_ROOT / ".runtime/tiktoken-cache"
encoding = load_o200k_base_offline(CACHE)   # cache가 없거나 hash가 다르면 여기서 실패한다
print(f"offline load OK  tokenizer {TOKENIZER_ID}  tiktoken {TIKTOKEN_VERSION}  "
      f"n_vocab {encoding.n_vocab:,}")

PROV_SQL = (
    "SELECT count(DISTINCT tokenizer_id) AS d_id,"
    "  count(DISTINCT tiktoken_version) AS d_ver,"
    "  count(DISTINCT encoding_file_sha256) AS d_enc,"
    "  count(DISTINCT mergeable_ranks_hash) AS d_ranks,"
    "  count(DISTINCT pat_str_sha256) AS d_pat,"
    "  count(DISTINCT special_tokens_hash) AS d_special,"
    "  count(DISTINCT tokenizer_config_sha256) AS d_cfg,"
    "  any_value(encoding_file_sha256) AS enc, any_value(mergeable_ranks_hash) AS ranks,"
    "  any_value(pat_str_sha256) AS pat, any_value(special_tokens_hash) AS special,"
    "  any_value(tokenizer_config_sha256) AS cfg"
    f" FROM {T}")
prov = CON.execute(PROV_SQL).fetchdf().iloc[0]
distinct_ok = all(int(prov[k]) == 1 for k in
                  ("d_id", "d_ver", "d_enc", "d_ranks", "d_pat", "d_special", "d_cfg"))
print(f"\ndistinct provenance values over {N:,} rows all equal 1: {distinct_ok}")
assert distinct_ok

pairs = [("encoding_file_sha256", prov["enc"], ENCODING_FILE_SHA256),
         ("mergeable_ranks_hash", prov["ranks"], MERGEABLE_RANKS_HASH),
         ("pat_str_sha256", prov["pat"], PAT_STR_SHA256),
         ("special_tokens_hash", prov["special"], SPECIAL_TOKENS_HASH),
         ("tokenizer_config_sha256", prov["cfg"], TOKENIZER_CONFIG_SHA256)]
for name, in_artifact, in_module in pairs:
    print(f"  {name:26s} {'MATCH' if in_artifact == in_module else 'MISMATCH'}  {in_artifact}")
assert all(a == b for _, a, b in pairs)
print("\nTOKENIZER_PROVENANCE_FREEZE = PASS")
print(f"  special tokens used : {TOKENIZER_CONFIG['special_tokens_used']}")
print(f"  chat template used  : {TOKENIZER_CONFIG['chat_template_used']}")
print(f"  network fallback    : {TOKENIZER_CONFIG['network_fallback_allowed']}")

offline load OK  tokenizer o200k_base  tiktoken 0.13.0  n_vocab 200,019



distinct provenance values over 3,835,988 rows all equal 1: True
  encoding_file_sha256       MATCH  446a9538cb6c348e3516120d7c08b09f57c36495e2acfffe59a5bf8b0cfb1a2d
  mergeable_ranks_hash       MATCH  f2f614601c635339047c0ec251d13afcfd8e3bc01440bca9ab0bdf17ed61e2d0
  pat_str_sha256             MATCH  2d1b8dc11e89af71459b36004f698ab3693f59fd84f63e8ec2b49564ab857420
  special_tokens_hash        MATCH  160541c3dd5153d72838e5770937be894f3decc367096bf849726a40d7afa14d
  tokenizer_config_sha256    MATCH  c29ea3894c203457564e7b49e4e8ae3a58acf0e351c6fc581c0a8734bbc41da0

TOKENIZER_PROVENANCE_FREEZE = PASS
  special tokens used : False
  chat template used  : False
  network fallback    : False


### Special tokens were never admitted

SSOT §14.2 forbids special tokens on Track A. Rather than asserting it, the maximum token id in the
population is compared against the mergeable-rank boundary: any special token would sit at or above
it.

In [5]:
MAXID_SQL = (
    "SELECT max(m) AS max_token_id FROM ("
    f"  SELECT max(x) AS m FROM {T}, unnest(ko_token_ids) AS t(x)"
    "  UNION ALL"
    f"  SELECT max(x) AS m FROM {T}, unnest(en_token_ids) AS t(x))")
max_token_id = int(CON.execute(MAXID_SQL).fetchone()[0])
# 경계는 private 속성에서 유도하지 않고 Phase-0에서 동결된 artifact manifest 값을 쓴다.
phase0 = json.loads((PROJECT_ROOT / "outputs/manifests/TOKENIZER_O200K_BASE_ARTIFACT_v001.json"
                     ).read_text(encoding="utf-8"))
n_mergeable = int(phase0["mergeable_ranks_count"])
n_special = int(phase0["special_tokens_count"])
print(f"  max token id in the population : {max_token_id:,}")
print(f"  mergeable ranks (Phase-0 freeze): {n_mergeable:,}")
print(f"  special tokens (Phase-0 freeze) : {n_special}")
print(f"  n_vocab (live encoder)          : {encoding.n_vocab:,}")
assert phase0["encoding_file_sha256"] == ENCODING_FILE_SHA256
assert max_token_id < n_mergeable, "special token id observed in Track A output"
print("\nNO_SPECIAL_TOKEN_ADMITTED = PASS (every id sits below the special-token range)")

  max token id in the population : 199,997
  mergeable ranks (Phase-0 freeze): 199,998
  special tokens (Phase-0 freeze) : 2
  n_vocab (live encoder)          : 200,019

NO_SPECIAL_TOKEN_ADMITTED = PASS (every id sits below the special-token range)


## 3 — SSOT §12.4 field conformance

In [6]:
D04_REQUIRED = ["measurement_id", "pair_id", "tokenizer_id", "tiktoken_version",
                "encoding_file_sha256", "mergeable_ranks_hash", "pat_str_sha256",
                "special_tokens_hash", "ko_token_ids", "en_token_ids", "ko_token_count",
                "en_token_count", "token_premium", "log_token_premium", "token_difference",
                "compression_penalty", "roundtrip_ok"]
schema = CON.execute(f"DESCRIBE SELECT * FROM {T}").fetchall()
present = {row[0]: row[1] for row in schema}
missing = [f for f in D04_REQUIRED if f not in present]
print(f"SSOT §12.4 required fields present: {len(D04_REQUIRED) - len(missing)}/{len(D04_REQUIRED)}"
      f"   missing {missing or '[]'}")
assert not missing
print(f"\nko_token_ids type : {present['ko_token_ids']}")
print(f"en_token_ids type : {present['en_token_ids']}")
print(f"total columns     : {len(schema)}")

SSOT §12.4 required fields present: 17/17   missing []

ko_token_ids type : INTEGER[]
en_token_ids type : INTEGER[]
total columns     : 28


## 4 — Roundtrip over every final pair (G3 core evidence)

SSOT §14.2(3) requires `decode(encode(text)) == text`. The stored flag is checked, and the token id
arrays are checked against the stored counts so the flag cannot be true over an empty or truncated
array.

In [7]:
RT_SQL = (
    "SELECT count(*) AS n,"
    "  sum((NOT roundtrip_ok)::INT) AS roundtrip_failures,"
    "  sum((length(ko_token_ids) <> ko_token_count)::INT) AS ko_len_mismatch,"
    "  sum((length(en_token_ids) <> en_token_count)::INT) AS en_len_mismatch,"
    "  sum((ko_token_count <= 0 OR en_token_count <= 0)::INT) AS nonpositive_counts,"
    "  sum((token_difference <> ko_token_count - en_token_count)::INT) AS token_diff_violation,"
    "  sum((abs(token_premium - ko_token_count::DOUBLE / en_token_count) > 1e-12)::INT)"
    "    AS tp_definition_violation"
    f" FROM {T}")
rt = CON.execute(RT_SQL).fetchdf().iloc[0]
for key, value in rt.items():
    print(f"  {key:24s} {int(value):,}")
pass_rate = 1 - int(rt["roundtrip_failures"]) / int(rt["n"])
print(f"\n  roundtrip pass rate      {pass_rate:.6f}")
assert all(int(rt[k]) == 0 for k in
           ("roundtrip_failures", "ko_len_mismatch", "en_len_mismatch", "nonpositive_counts",
            "token_diff_violation", "tp_definition_violation"))
print("ROUNDTRIP_100_PERCENT = PASS")

  n                        3,835,988
  roundtrip_failures       0
  ko_len_mismatch          0
  en_len_mismatch          0
  nonpositive_counts       0
  token_diff_violation     0
  tp_definition_violation  0

  roundtrip pass rate      1.000000
ROUNDTRIP_100_PERCENT = PASS


## 5 — Exact decomposition over every final pair (G2 core evidence)

Recomputed **from the constituent fields** rather than read from `identity_abs_error`, so the check
does not depend on the value the pipeline stored. The frozen tolerance is `epsilon = 1e-10`
(D-RD-01).

In [8]:
from tokenization_premium.tokenizer_measurement import EXACT_DECOMPOSITION_EPSILON

DECOMP_SQL = (
    "WITH recalc AS ("
    "  SELECT abs(log_token_premium -"
    "             (log_code_point_ratio + log_byte_density_ratio + log_compression_penalty))"
    "         AS err,"
    "         abs(log_token_premium - ln(token_premium)) AS logtp_err,"
    "         abs(token_premium - code_point_ratio * byte_density_ratio * compression_penalty)"
    "         / token_premium AS multiplicative_rel_err"
    f"  FROM {T})"
    " SELECT count(*) AS checked,"
    f"  sum((err >= {EXACT_DECOMPOSITION_EPSILON})::INT) AS violations,"
    "  max(err) AS max_abs_error,"
    "  quantile_cont(err, 0.99) AS p99, quantile_cont(err, 0.999) AS p999,"
    "  max(logtp_err) AS max_logtp_consistency_error,"
    "  max(multiplicative_rel_err) AS max_multiplicative_rel_error"
    " FROM recalc")
dec = CON.execute(DECOMP_SQL).fetchdf().iloc[0]
print(f"  identity   |logTP - (logCR + logBDR + logCP)| < {EXACT_DECOMPOSITION_EPSILON}")
print(f"  checked    {int(dec['checked']):,}")
print(f"  violations {int(dec['violations'])}")
print(f"  max abs error {dec['max_abs_error']:.6e}")
print(f"  p99 / p99.9   {dec['p99']:.6e} / {dec['p999']:.6e}")
print(f"  max |logTP - ln(TP)|                     {dec['max_logtp_consistency_error']:.6e}")
print(f"  max relative error of the product form   {dec['max_multiplicative_rel_error']:.6e}")
assert int(dec["violations"]) == 0
print("\nEXACT_DECOMPOSITION = PASS (0 violations over the full cohort)")

  identity   |logTP - (logCR + logBDR + logCP)| < 1e-10
  checked    3,835,988
  violations 0
  max abs error 8.881784e-16
  p99 / p99.9   3.330669e-16 / 3.885781e-16
  max |logTP - ln(TP)|                     0.000000e+00
  max relative error of the product form   6.481302e-16

EXACT_DECOMPOSITION = PASS (0 violations over the full cohort)


## 6 — Distribution of the estimand inputs

Descriptive only. No hypothesis test, no confidence interval, no claim about `median(logTP)` — that
is RQ1 and belongs to NB08.

In [9]:
FEATURES = ["ko_token_count", "en_token_count", "token_premium", "log_token_premium",
            "token_difference", "code_point_ratio", "byte_density_ratio", "compression_penalty",
            "ko_tokens_per_byte", "en_tokens_per_byte"]
sel = ", ".join(
    f'min({f}) AS "{f}|min", quantile_cont({f}, 0.25) AS "{f}|p25", '
    f'median({f}) AS "{f}|median", quantile_cont({f}, 0.75) AS "{f}|p75", '
    f'quantile_cont({f}, 0.99) AS "{f}|p99", max({f}) AS "{f}|max", avg({f}) AS "{f}|mean"'
    for f in FEATURES)
raw = CON.execute(f"SELECT {sel} FROM {T}").fetchdf().iloc[0]
profile = pd.DataFrame(
    [{"feature": f, **{stat: float(raw[f"{f}|{stat}"])
                       for stat in ("min", "p25", "median", "p75", "p99", "max", "mean")}}
     for f in FEATURES]).set_index("feature")
totals = CON.execute(
    f"SELECT sum(ko_token_count) AS ko, sum(en_token_count) AS en FROM {T}").fetchdf().iloc[0]
print(f"total tokens   KO {int(totals['ko']):,}   EN {int(totals['en']):,}")
profile.round(6)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

total tokens   KO 104,042,094   EN 77,374,146


,min,p25,median,p75,p99,max,mean
feature,,,,,,,
ko_token_count,1.000000,14.000000,21.000000,39.000000,75.000000,401.000000,27.122633
en_token_count,1.000000,10.000000,16.000000,28.000000,58.000000,453.000000,20.170591
token_premium,0.064018,1.166667,1.333333,1.533333,2.250000,38.000000,1.362988
log_token_premium,-2.748596,0.154151,0.287682,0.427444,0.810930,3.637586,0.285177
token_difference,-424.000000,2.000000,5.000000,10.000000,28.000000,154.000000,6.952042
code_point_ratio,0.022192,0.408451,0.467456,0.538462,0.833333,60.000000,0.482697
byte_density_ratio,0.666667,2.353846,2.450000,2.500000,2.625000,3.000000,2.412208
compression_penalty,0.214286,1.042453,1.191430,1.336590,1.741821,5.142857,1.194266
ko_tokens_per_byte,0.090909,0.223529,0.245614,0.272727,0.444444,1.000000,0.253869


## 7 — Token-byte audit sample (SSOT §31 G3)

G3 requires an audit sample with token-byte inspection. The sample is local-only and stores decoded
token bytes as hex, never raw sentence text.

In [10]:
audit_manifest = json.loads(
    (PROJECT_ROOT / "outputs/manifests/TOKEN_AUDIT_SAMPLE_MANIFEST_v001.json"
     ).read_text(encoding="utf-8"))
print(f"  sample n              {audit_manifest['n']}")
print(f"  roundtrip all ok      {audit_manifest['roundtrip_ok_all']}")
print(f"  max identity error    {audit_manifest['max_identity_abs_error']:.3e}")
print(f"  raw text included     {audit_manifest['raw_text_included']}")
print(f"  token byte encoding   {audit_manifest['token_bytes_encoding']}")
print(f"  reasons               {audit_manifest['sample_reason_counts']}")
print(f"  coverage domain       {audit_manifest['coverage']['domain']}")
assert audit_manifest["roundtrip_ok_all"] and not audit_manifest["raw_text_included"]
print("\nTOKEN_BYTE_AUDIT_SAMPLE = PRESENT")

  sample n              100
  roundtrip all ok      True
  max identity error    8.882e-16
  raw text included     False
  token byte encoding   decode_single_token_bytes -> hex, head 32 tokens per side
  reasons               {'domain_dialogue': 10, 'domain_general': 10, 'domain_other': 10, 'domain_technology': 10, 'longest': 10, 'mixed_script': 10, 'rare_unicode': 10, 'shortest': 10, 'tp_highest': 10, 'tp_lowest': 10}
  coverage domain       {'dialogue': 15, 'general': 44, 'other': 31, 'technology': 10}

TOKEN_BYTE_AUDIT_SAMPLE = PRESENT


## 8 — Canonical summary

In [11]:
summary = {
    "notebook": "notebooks/05_o200k_measurement.ipynb",
    "phase": "Phase 4 — D-04 Token Measurement",
    "canonical_artifact": {k: TOK[k] for k in
                           ("name", "path", "sha256", "row_count", "column_count", "pair_set_md5")},
    "supersedes": "synthetic-only 4-row scaffold with the full population deferred",
    "tokenizer_provenance_freeze": "PASS",
    "no_special_token_admitted": "PASS",
    "ssot_12_4_field_conformance": "PASS",
    "roundtrip_pass_rate": pass_rate,
    "exact_decomposition": {"epsilon": EXACT_DECOMPOSITION_EPSILON,
                            "checked": int(dec["checked"]),
                            "violations": int(dec["violations"]),
                            "max_abs_error": float(dec["max_abs_error"])},
    "gate": ("G3_TOKENIZER_INTEGRITY_PASS and the G2 exact-decomposition evidence, adjudicated by "
             f"Claude-B at {ADJUDICATION_COMMIT[:12]}; this notebook reproduces the evidence, "
             "it does not re-adjudicate the gates"),
    "inference_drawn": False,
    "rebuild_performed": False,
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
CON.close()

{
  "notebook": "notebooks/05_o200k_measurement.ipynb",
  "phase": "Phase 4 — D-04 Token Measurement",
  "canonical_artifact": {
    "name": "TOKEN_O200K_BASE_v001",
    "path": "data/registry/TOKEN_O200K_BASE_v001.parquet",
    "sha256": "1c30e3276222dd94885ae4f79fc6ab5c45e4e26226de0afd91fc6b1f7d2c16e7",
    "row_count": 3835988,
    "column_count": 28,
    "pair_set_md5": "d9660d654ee449e4d0c23a0070225274"
  },
  "supersedes": "synthetic-only 4-row scaffold with the full population deferred",
  "tokenizer_provenance_freeze": "PASS",
  "no_special_token_admitted": "PASS",
  "ssot_12_4_field_conformance": "PASS",
  "roundtrip_pass_rate": 1.0,
  "exact_decomposition": {
    "epsilon": 1e-10,
    "checked": 3835988,
    "violations": 0,
    "max_abs_error": 8.881784197001252e-16
  },
  "gate": "G3_TOKENIZER_INTEGRITY_PASS and the G2 exact-decomposition evidence, adjudicated by Claude-B at 441d5802bfeb; this notebook reproduces the evidence, it does not re-adjudicate the gates",
  "infere

---

`TOKEN_O200K_BASE_v001` is the canonical D-04 artifact. The regex chunk mechanism (NB06) and the RQ1
inference (NB08) are separate phases and are not started here.